In [29]:
!pip install pandas numpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import uuid
import json
from datetime import datetime, timedelta


class Application:
    DEFAULT_PERMS = ["Read", "Write", "Execute", "Approve", "Audit", "Report"]

    def __init__(self, name, perms=None):
        self.name = name
        self.permissions = {}
        for p in (perms or Application.DEFAULT_PERMS): # ignore this because of role inheritance
            perm = Permission(p, self)
            setattr(self, p, perm) # self.Read, self.Write, etc... = Permission(...)
            self.permissions[p] = perm


class Permission:
    def __init__(self, action, application):
        self.name = application.name + '.' + action
        self.action = action
        self.application = application

class Role:
    def __init__(self, name, parent=None):
        self.name = name
        self.permissions = []
        self.parent = parent

    def add_permission(self, permission):
        self.permissions.append(permission)
    def add_permissions(self, permissions):
        self.permissions.extend(permissions)

    def get_all_permissions(self):
        perms = list(self.permissions) #changed from set to list
        if self.parent:
            perms.extend(self.parent.get_all_permissions())
        return list(set(perms)) # clears up any duplicates


class User:
    def __init__(self):
        self.id = str(uuid.uuid4())
        self.roles = []
        self.active_role = None
        self.session_start = None

    def assign_role(self, role):
        if len(self.roles) < 4:
            self.roles.append(role)
        else: 
            print(f"Error: {self.id} already has 4 roles.")

    def get_active_role(self):
        return self.active_role

    def start_session(self, role_name):
        for role in self.roles: # User has to have role in order to impersonate
            if role.name == role_name: # If user has role, then...
                self.active_role = role
                self.session_start = datetime.now()
                print(f"{self.id} starts session with Role {role.name}")
                return

        print(f"ERROR: {self.id} cannot use role {role_name}.")


    def log_attempt(self, app_name, action, result):
        #log to json file
        log_entry = {
            "timestamp" : datetime.now().strftime("%a, %d %b %Y %H:%M:%S GMT"),
            "user" : self.id, 
            "application" : app_name, 
            "access_type" : action, 
            "result" : result
        }
        with open("access_log.json", "a") as file:
            file. write(json.dumps(log_entry) + "\n")

    def check_access(self, app_name, action):
        if not self.active_role:
            print(f"Access Denied: {self.id} -> No active session.")
            self.log_attempt(app_name, action, "Denied - No Active Session")
            return

        if datetime.now() - self.session_start > timedelta(hours=1):
            print(f"ERROR: Session expired for {self.id}.")
            self.log_attempt(app_name, action, "Denied - Session Expired")
            return

        all_perms = self.active_role.get_all_permissions()
        for perm in all_perms:
            if perm.application.name == app_name and perm.action == action:
                print(f"Access Granted: {self.id} [Role {self.active_role.name}] -> {app_name} ({action})")
                self.log_attempt(app_name, action, "Granted")
                return



        print(f"Access Denied: {self.id} [Role {self.active_role.name}] -> {app_name} ({action})")
        self.log_attempt(app_name, action, "Denied")


In [31]:
# SEEDING

#the tables need to be in here as well
# Create applications
print("=== Seeding Applications... ===")
mmi = Application("Money Market Instruments")
derivatives = Application("Derivatives Trading")
interest = Application("Interest Instruments")
pci = Application("Private Consumer Instruments")
ibs = Application("Investment Banking Suite")
tm = Application("Treasury Management")
lm = Application("Loan Management")
rms = Application("Risk Management System")
ctp = Application("Corporate Trading Platform")
sms = Application("Share Management System")
ecg = Application("E-Commerce Gateway")
obap = Application("Office Banking Admin Panel")


# Roles
print("=== Seeding Roles... ===")
role_a = Role('A')
role_b = Role('B', parent=role_a)
role_c = Role('C', parent=role_b)
role_d = Role('D', parent=role_c)
role_e = Role('E', parent=role_d)
role_f = Role('F', parent=role_e)
role_g = Role('G')
role_h = Role('H')
role_i = Role('I')

print("=== Seeding Role Permissions... ===")

role_a.add_permissions([mmi.Read, mmi.Write, derivatives.Read, derivatives.Execute, interest.Read, interest.Execute])
role_b.add_permissions([mmi.Execute, derivatives.Write, pci.Read, pci.Write])
role_c.add_permissions([ibs.Read, ibs.Write, ibs.Approve, tm.Read, tm.Write, tm.Execute])
role_d.add_permissions([lm.Read, lm.Write, lm.Execute])
role_e.add_permissions([rms.Read, rms.Write, rms.Execute, rms.Approve])
role_f.add_permissions([ctp.Read, ctp.Write, ctp.Execute, ctp.Audit, ctp.Report])
role_g.add_permissions([sms.Read, sms.Write])
role_h.add_permissions([ecg.Read, ecg.Write, ecg.Execute])
role_i.add_permissions([obap.Read, obap.Write, obap.Approve, obap.Audit])

print("=== Seeding Complete... ===")


# Scenario 1: Successful Inheritance
print("User 1 with Role C should be able to RWX MMI from role B and A which is inherited")
u1 = User()
u1.assign_role(role_c)
u1.start_session("C")
u1.check_access('Money Market Instruments', 'Read') # Inherited from Role A

# Scenario 2: Access Denied (Unauthorized)
print("User 1 should not be able to access applications in which they do not have permissions for, such as RMS.*")
u1.check_access('Risk Management System', 'Approve')

# Scenario 3: Unauthorized Role Selection
print("User 2 with Role G should not be able to start session with Role A since they don't have it")
u2 = User()
u2.assign_role(role_g)
u2.start_session("A") # Bob doesn't have Role A

# Scenario 4: Attempt to Access without an active session
print("User 3 with Role A should not be able to access MMI.Read without starting a session")
u3 = User()
u3.assign_role(role_a)
# u3.start_session("B")
u3.check_access('Money Market Instruments', 'Read') # No active session

# Scenario 5: Session Expires
print("User 4 with Role B should be denied access due to their session expiring.")
u4 = User()
u4.assign_role(role_b)
u4.start_session("B")
# Simulate session expiry by manually setting session_start to 2 hours ago
u4.session_start -= timedelta(hours=2)
u4.check_access('Derivatives Trading', 'Write')



=== Seeding Applications... ===
=== Seeding Roles... ===
=== Seeding Role Permissions... ===
=== Seeding Complete... ===
User 1 with Role C should be able to RWX MMI from role B and A which is inherited
df798c11-ee41-4397-a1cb-ec71a8f637c3 starts session with Role C
Access Granted: df798c11-ee41-4397-a1cb-ec71a8f637c3 [Role C] -> Money Market Instruments (Read)
User 1 should not be able to access applications in which they do not have permissions for, such as RMS.*
Access Denied: df798c11-ee41-4397-a1cb-ec71a8f637c3 [Role C] -> Risk Management System (Approve)
User 2 with Role G should not be able to start session with Role A since they don't have it
ERROR: f1fe2e2d-a0b7-42ae-a90c-0a1f17827ac3 cannot use role A.
User 3 with Role A should not be able to access MMI.Read without starting a session
Access Denied: df41d47b-90c0-43c2-98c0-bfe392a7c2e6 -> No active session.
User 4 with Role B should be denied access due to their session expiring.
30dcca99-8237-4c1d-8f2e-76430fd34321 starts se